# ЮрТэг — подготовка датасета на Kaggle

**Что делает этот ноутбук:**
1. Генерирует синтетические документы через 7B для редких типов
2. Перемечает реальные документы через 7B (заменяет GLM-метки)
3. Объединяет всё и делает train/val split

**Входные данные (добавь как Kaggle Datasets):**
- `yurteg-7b` — GGUF файл модели
- `yurteg-dataset` — labeled_data_normalized.jsonl + synthetic_sft.jsonl

In [ ]:
GGUF_PATH = "/kaggle/input/yurteg-7b-model/qwen2.5-7b-instruct.Q4_K_M.gguf"
REAL_DATA_PATH = "/kaggle/input/yurteg-dataset/labeled_data_normalized.jsonl"
SYNTHETIC_PATH = "/kaggle/input/yurteg-dataset/synthetic_sft.jsonl"
OUTPUT_DIR = "/kaggle/working"

In [ ]:
import subprocess, sys

# Проверяем версию CUDA
result = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print(result.stdout)

# Ставим готовый wheel под CUDA 12.1 (Kaggle T4)
!pip install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
    --no-cache-dir -q

!pip install tqdm -q

In [ ]:
from llama_cpp import Llama
from tqdm import tqdm
import json, random, hashlib, os
from datetime import datetime
from collections import Counter
from pathlib import Path

random.seed(42)

llm = Llama(
    model_path=GGUF_PATH,
    n_gpu_layers=-1,
    n_ctx=2048,
    verbose=False,
)
print("Модель загружена")

## Общие утилиты

In [ ]:
TRAINING_SYSTEM = """Ты — опытный юрист-аналитик. Извлеки структурированные метаданные из юридического документа.

ПРАВИЛА:
1. Отвечай СТРОГО чистым JSON. Без текста до/после, без обёрток ```json```.
2. Отсутствующую информацию ставь null (не пустую строку "").
3. Списки всегда массивы: parties=[], special_conditions=[].
4. confidence — число от 0.0 до 1.0.
5. Сумму пиши с пробелами-разделителями и валютой: "1 500 000 руб.".
6. Даты строго YYYY-MM-DD.
7. ФИО в именительном падеже: "Иванов Иван Иванович".
8. Формат ИП: "ИП Фамилия Имя Отчество"."""

TRAINING_USER = """Извлеки метаданные из текста юридического документа.

Верни JSON с полями: document_type, counterparty, subject, date_signed, date_start, date_end, amount, special_conditions, parties, confidence, is_template

Текст документа:
{text}"""

GENERATION_SYSTEM = """Ты — опытный юрист, составляющий юридические документы.
Генерируй реалистичные российские юридические документы с конкретными данными.
Документы должны быть полными, профессиональными, на русском языке."""

GENERATION_USER = """Составь реалистичный юридический документ типа "{doc_type}".

Используй эти данные:
- Сторона 1: {party1}
- Сторона 2: {party2}
- Город: {city}
- Год: {year}
- Сумма (если применимо): {amount}

Требования:
- Полный текст документа с номером, датой, реквизитами сторон
- 3-7 разделов с конкретными условиями
- Только текст документа, без пояснений

Документ:"""

GENERATION_USER_KARTOCHKA = """Составь реалистичную Карточку контрагента.

ВАЖНО: Карточка — это НЕ договор. Это анкета с реквизитами одной организации:
наименование, ИНН (10 цифр для юрлица, 12 для ИП), КПП (9 цифр, только юрлица),
ОГРН (13 цифр), ОГРНИП (15 цифр для ИП), адрес, банковские реквизиты, контакты.
Никаких сторон и обязательств.

Контрагент: {party1}
Город: {city}
Год: {year}

Карточка контрагента:"""

COMPANIES = [
    "ООО «Ромашка»", "ООО «Альфа-Строй»", "ООО «ТехноГрупп»",
    "АО «Северсталь»", "ООО «Медиасервис»", "ПАО «РосТех»",
    "ООО «Логистик Про»", "ООО «СтройКомплект»", "АО «ФинансГрупп»",
    "ООО «Диджитал Солюшнс»", "ООО «АгроТрейд»", "ООО «МеталлТорг»",
]
INDIVIDUALS = [
    "ИП Петров Алексей Сергеевич", "ИП Сидорова Наталья Владимировна",
    "ИП Козлов Дмитрий Андреевич", "ИП Новикова Елена Михайловна",
    "ИП Морозов Игорь Николаевич", "ИП Волкова Анна Юрьевна",
]
CITIES = ["г. Москва", "г. Санкт-Петербург", "г. Екатеринбург", "г. Новосибирск", "г. Казань"]
AMOUNTS = ["500 000 руб.", "1 200 000 руб.", "250 000 руб.", "3 500 000 руб.", "75 000 руб."]

def make_id(text):
    return hashlib.md5(text.encode()).hexdigest()[:16]

def clean_metadata(meta):
    if "is_template" not in meta:
        meta["is_template"] = False
    if not isinstance(meta.get("parties"), list):
        meta["parties"] = []
    if not isinstance(meta.get("special_conditions"), list):
        meta["special_conditions"] = []
    for f in ["counterparty", "subject", "amount", "date_signed", "date_start", "date_end"]:
        if meta.get(f) == "":
            meta[f] = None
    for f in ["date_signed", "date_start", "date_end"]:
        val = meta.get(f)
        if val and ("00-00" in str(val) or len(str(val)) != 10):
            meta[f] = None
    return meta

def llm_call(system, user, max_tokens=800):
    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0,
        max_tokens=max_tokens,
    )
    return resp["choices"][0]["message"]["content"].strip()

def parse_json_response(content):
    """Парсим JSON из ответа модели."""
    text = content
    if "```" in text:
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    brace = text.find("{")
    if brace != -1:
        text = text[brace:]
    return json.loads(text.strip())

print("Утилиты загружены")

## Шаг 1 — Генерация синтетических документов

In [ ]:
# Типы которых не хватает — добираем до целевого количества
TARGETS = {
    "Политика обработки ПД": 15,
    "Правила акции": 15,
    "Оферта на оказание услуг": 12,
    "Рамочный договор": 7,
    "Договор дарения": 6,
    "Договор безвозмездного пользования": 8,
}

# Описания редких типов — вставляются в промпт чтобы 7B понял что генерировать
TYPE_DESCRIPTIONS = {
    "Политика обработки ПД": (
        "Внутренний документ организации, описывающий как и зачем компания собирает, "
        "хранит и обрабатывает персональные данные клиентов и сотрудников. "
        "Содержит: цели обработки, правовые основания, перечень данных, сроки хранения, "
        "права субъектов ПД, меры защиты. Публикуется на сайте компании."
    ),
    "Правила акции": (
        "Документ организатора маркетинговой акции или розыгрыша призов. "
        "Содержит: название акции, организатор, период проведения, территория, "
        "условия участия, призовой фонд, порядок определения победителей, "
        "порядок вручения призов, ограничения по возрасту."
    ),
    "Оферта на оказание услуг": (
        "Публичное предложение заключить договор на определённых условиях, "
        "адресованное неограниченному кругу лиц. Содержит: описание услуги, цену, "
        "порядок акцепта (принятия оферты), срок действия оферты, реквизиты оферента. "
        "Акцепт = оплата или подписание согласия."
    ),
    "Рамочный договор": (
        "Договор, определяющий общие условия долгосрочного сотрудничества. "
        "Сам не порождает обязательств по конкретной сделке — "
        "отдельные поставки или услуги оформляются заявками или спецификациями. "
        "Содержит: общие условия, порядок оформления заявок, ответственность сторон."
    ),
    "Договор дарения": (
        "Договор по которому даритель безвозмездно передаёт одаряемому имущество. "
        "Содержит: описание предмета дара, его стоимость, условия и момент передачи, "
        "права и обязанности сторон. Для недвижимости требует нотариального удостоверения."
    ),
    "Договор безвозмездного пользования": (
        "Договор ссуды — передача имущества во временное пользование без оплаты. "
        "Ссудодатель передаёт вещь ссудополучателю, который обязан вернуть её в том же состоянии. "
        "Содержит: описание имущества, срок пользования, обязанности по содержанию, "
        "ответственность за ущерб."
    ),
}

GENERATION_USER_WITH_DESC = """Составь реалистичный юридический документ типа "{doc_type}".

Описание: {description}

Используй эти данные:
- Сторона 1: {party1}
- Сторона 2: {party2}
- Город: {city}
- Год: {year}
- Сумма (если применимо): {amount}

Требования:
- Полный текст документа с номером, датой, реквизитами сторон
- 3-7 разделов с конкретными условиями
- Только текст документа, без пояснений

Документ:"""

def generate_doc(doc_type):
    """Генерирует текст документа через 7B."""
    party1 = random.choice(COMPANIES + INDIVIDUALS)
    party2 = random.choice(COMPANIES + INDIVIDUALS)
    while party2 == party1:
        party2 = random.choice(COMPANIES + INDIVIDUALS)

    if doc_type == "Карточка контрагента":
        prompt = GENERATION_USER_KARTOCHKA.format(
            party1=party1, city=random.choice(CITIES), year=random.choice([2023, 2024, 2025])
        )
    elif doc_type in TYPE_DESCRIPTIONS:
        prompt = GENERATION_USER_WITH_DESC.format(
            doc_type=doc_type,
            description=TYPE_DESCRIPTIONS[doc_type],
            party1=party1, party2=party2,
            city=random.choice(CITIES), year=random.choice([2023, 2024, 2025]),
            amount=random.choice(AMOUNTS),
        )
    else:
        prompt = GENERATION_USER.format(
            doc_type=doc_type, party1=party1, party2=party2,
            city=random.choice(CITIES), year=random.choice([2023, 2024, 2025]),
            amount=random.choice(AMOUNTS),
        )
    return llm_call(GENERATION_SYSTEM, prompt, max_tokens=1500)

def extract_metadata(text):
    """Извлекает JSON метаданные из текста."""
    prompt = TRAINING_USER.format(text=text[:3000])
    content = llm_call(TRAINING_SYSTEM, prompt, max_tokens=600)
    meta = parse_json_response(content)
    return clean_metadata(meta)

def make_sft_record(doc_text, metadata, doc_type):
    user_content = TRAINING_USER.format(text=doc_text[:6000])
    return {
        "id": make_id(doc_text),
        "source": "synthetic_7b",
        "doc_type": doc_type,
        "generated_at": datetime.now().isoformat(),
        "messages": [
            {"role": "system", "content": TRAINING_SYSTEM},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": json.dumps(metadata, ensure_ascii=False)},
        ]
    }

In [ ]:
synthetic_new = []

for doc_type, count in TARGETS.items():
    generated = 0
    failed = 0
    pbar = tqdm(total=count, desc=doc_type[:30], unit="doc")

    while generated < count:
        try:
            doc_text = generate_doc(doc_type)
            if not doc_text or len(doc_text) < 200:
                failed += 1
                continue

            meta = extract_metadata(doc_text)
            if not meta.get("document_type"):
                failed += 1
                continue

            # Принимаем если тип совпадает или близкий
            meta["document_type"] = doc_type
            record = make_sft_record(doc_text, meta, doc_type)
            synthetic_new.append(record)
            generated += 1
            pbar.update(1)

        except Exception as e:
            failed += 1
            if failed > count * 3:
                print(f"  Слишком много ошибок, пропускаем {doc_type}")
                break

    pbar.close()

print(f"\nНовая синтетика: {len(synthetic_new)} документов")

## Шаг 2 — Разметка реальных документов через 7B

In [ ]:
real_records_raw = []
with open(REAL_DATA_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            real_records_raw.append(json.loads(line))
print(f"Реальных документов: {len(real_records_raw)}")

In [ ]:
real_records_relabeled = []
replaced = 0
kept = 0

for record in tqdm(real_records_raw, desc="Разметка реальных", unit="doc"):
    input_text = record.get("input_text", "").strip()

    orig_output = record.get("output", {})
    if isinstance(orig_output, str):
        try:
            orig_output = json.loads(orig_output)
        except Exception:
            orig_output = {}

    new_output = clean_metadata(dict(orig_output))
    label_source = "glm_original"

    if input_text:
        try:
            meta = extract_metadata(input_text)
            if meta.get("document_type"):
                new_output = meta
                label_source = "yurteg_7b"
                replaced += 1
            else:
                kept += 1
        except Exception:
            kept += 1

    # Формируем SFT-запись
    user_content = TRAINING_USER.format(text=input_text[:6000])
    real_records_relabeled.append({
        "id": record.get("id") or make_id(input_text),
        "source": label_source,
        "doc_type": new_output.get("document_type", ""),
        "messages": [
            {"role": "system", "content": TRAINING_SYSTEM},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": json.dumps(new_output, ensure_ascii=False)},
        ]
    })

print(f"\nЗаменено (7B): {replaced}, оставлено (GLM): {kept}")

## Шаг 3 — Объединение и train/val split

In [ ]:
import random

# Загружаем старую синтетику (накопленную локально)
synthetic_old = []
try:
    with open(SYNTHETIC_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                synthetic_old.append(json.loads(line))
    print(f"Старая синтетика: {len(synthetic_old)}")
except FileNotFoundError:
    print("synthetic_sft.jsonl не найден, пропускаем")

# Объединяем всё
all_records = real_records_relabeled + synthetic_old + synthetic_new

# Дедупликация по id
seen = set()
deduped = []
for r in all_records:
    rid = r.get("id") or make_id(str(r["messages"]))
    if rid not in seen:
        seen.add(rid)
        deduped.append(r)

print(f"Итого после дедупликации: {len(deduped)}")

# Статистика
counts = Counter(r.get("doc_type", "?") for r in deduped)
print(f"Типов: {len(counts)}")
print("Топ-10:")
for t, c in counts.most_common(10):
    print(f"  {c:4d}  {t}")

In [ ]:
random.seed(42)
random.shuffle(deduped)

val_size = max(1, int(len(deduped) * 0.1))
val_records = deduped[:val_size]
train_records = deduped[val_size:]

print(f"Train: {len(train_records)}, Val: {len(val_records)}")

train_path = f"{OUTPUT_DIR}/train.jsonl"
val_path = f"{OUTPUT_DIR}/val.jsonl"

with open(train_path, "w", encoding="utf-8") as f:
    for r in train_records:
        f.write(json.dumps({"messages": r["messages"]}, ensure_ascii=False) + "\n")

with open(val_path, "w", encoding="utf-8") as f:
    for r in val_records:
        f.write(json.dumps({"messages": r["messages"]}, ensure_ascii=False) + "\n")

print(f"\nГотово!")
print(f"  {train_path}")
print(f"  {val_path}")